In [4]:
   import pandas as pd
   import requests
   from dotenv import load_dotenv
   import os
   import time

In [5]:
load_dotenv()
APP_ID = os.getenv("ADZUNA_APP_ID")
APP_KEY = os.getenv("ADZUNA_APP_KEY")
RESULTS_PER_PAGE = 50
MAX_PAGES = 10
all_dfs = []

for page in range (1, MAX_PAGES + 1):
    response = requests.get(
        "https://api.adzuna.com/v1/api/jobs/ch/search/1",
        params={"app_id": APP_ID, "app_key": APP_KEY, "results_per_page": RESULTS_PER_PAGE}
    )
    print(f"Page {page}: status {response.status_code}")
    time.sleep(1)
    if response.status_code != 200:
        print(f"Page {page} failed with status code {response.status_code}")
        break

    data = response.json()
    page_df=pd.DataFrame(data["results"])

    if page_df.empty:
        print(f"Page {page} returned no results.")
        break

    all_dfs.append(page_df)
    print(f"In Page {page} {len(page_df)} jobs were found.")
    time.sleep(1)

df_all=pd.concat(all_dfs, ignore_index=True)
print(f"Total {len(df_all)} jobs were found in {page} pages.")

df_all.head()
def check_path(path):
    directory = os.path.dirname(path)
    if not os.path.exists(directory):
        print(f"❌ Klasör yok: {directory}")
        return False
    if not os.access(directory, os.W_OK):
        print(f"❌ Yazma izni yok: {directory}")
        return False
    print(f"✅ OK: {path}")
    return True

if check_path("../data/raw/adzuna_jobs_raw.csv"):
    df_all.to_csv("../data/raw/adzuna_jobs_raw.csv", index=False)


Page 1: status 200
In Page 1 50 jobs were found.
Page 2: status 200
In Page 2 50 jobs were found.
Page 3: status 200
In Page 3 50 jobs were found.
Page 4: status 200
In Page 4 50 jobs were found.
Page 5: status 200
In Page 5 50 jobs were found.
Page 6: status 200
In Page 6 50 jobs were found.
Page 7: status 200
In Page 7 50 jobs were found.
Page 8: status 200
In Page 8 50 jobs were found.
Page 9: status 200
In Page 9 50 jobs were found.
Page 10: status 200
In Page 10 50 jobs were found.
Total 500 jobs were found in 10 pages.
✅ OK: ../data/raw/adzuna_jobs_raw.csv


In [3]:
target_dir = "../data/raw"

if not os.path.exists(target_dir):
    print(f"❌ Klasör yok: {target_dir}")
elif not os.access(target_dir, os.W_OK):
    print(f"❌ Klasör var ama yazma izni yok: {target_dir}")
else:
    print(f"✅ OK, buraya yazılabilir: {target_dir}")

✅ OK, buraya yazılabilir: ../data/raw


In [8]:
   data = response.json()
   print(data["count"])
   df = pd.DataFrame(data["results"])
   # df.head()
   # df.info()
   # df.describe()
   df["contract_type"].value_counts(dropna=False)

   keywords =["student", "intern", "internship", "trainee","werkstudent","praktikant"]
   mask = df["title"].str.lower().str.contains("|".join(keywords), na=False)
   working_students_df = df[mask]
   working_students_df[["title", "company", "contract_type"]]

75426


,title,company,contract_type


In [4]:
print(df["company"].iloc[0])
print(df["location"].iloc[0])
print(df["category"].iloc[0])

{'display_name': 'Albedis', '__CLASS__': 'Adzuna::API::Response::Company'}
{'display_name': 'Lausanne', 'area': ['Schweiz', 'Kanton Waadt', 'Lausanne', 'Lausanne'], '__CLASS__': 'Adzuna::API::Response::Location'}
{'__CLASS__': 'Adzuna::API::Response::Category', 'label': 'Stellen aus Handel & Bau', 'tag': 'trade-construction-jobs'}


In [5]:
df_export = df.copy()

df_export["company_name"] = df_export["company"].apply(
    lambda x: x.get("display_name") if isinstance(x, dict) else None
)
df_export["location_name"] = df_export["location"].apply(
    lambda x: x.get("display_name") if isinstance(x, dict) else None
)
df_export["category_label"] = df_export["category"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)

df_export.to_csv("../data/processed/adzuna_export.csv", index=False)
df_export.to_excel("../data/processed/adzuna_export.xlsx", index=False, engine="openpyxl")